# LNPHub In Vitro Expression 3-Class Subset

This notebook creates a study-aware in vitro expression subset from the full `lnphub_master_v2.0.0.csv` file. It keeps studies with more than 500 unique formulations, harmonizes delivery into three ordered value classes, saves the subset, and prints/plots per-study class statistics.


## 0. Clone Private Repository

This clones or updates the private `Maburidi/lnphub-submission` repository from the `dev` branch.


In [ ]:
# Cell 1 - Clone private LNPHub repo from dev branch
from pathlib import Path
from getpass import getpass
import os
import subprocess

REPO_URL = "https://github.com/Maburidi/lnphub-submission.git"
BRANCH = "dev"
PROJECT_DIR = Path("/content/lnphub-submission")


def get_github_token():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata
        token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        token = ""
    return token or getpass("GitHub token for the private repo: ").strip()


def run(cmd, token=None):
    printable = " ".join(map(str, cmd))
    if token:
        printable = printable.replace(token, "***TOKEN***")
    print("$", printable)
    res = subprocess.run(cmd, text=True, capture_output=True)
    out = (res.stdout + res.stderr).replace(token or "", "***TOKEN***")
    if out.strip():
        print(out[-4000:])
    if res.returncode != 0:
        raise RuntimeError(f"Command failed with code {res.returncode}")
    return res


token = get_github_token()
auth_repo_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

if PROJECT_DIR.exists():
    print(f"Repo already exists: {PROJECT_DIR}")
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", auth_repo_url], token=token)
    run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", auth_repo_url, str(PROJECT_DIR)], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())
run(["git", "-C", str(PROJECT_DIR), "branch", "--show-current"])


## 1. Load Full LNPHub Data


In [ ]:
# Cell 2 - Imports and paths
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

PROJECT_DIR = Path("/content/lnphub-submission")
DATA_DIR = PROJECT_DIR / "data_subsets"
OUT_DIR = PROJECT_DIR / "data_subsets" / "in_vitro_expression_3class"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = [
    DATA_DIR / "lnphub_master_v2.0.0.csv",
    PROJECT_DIR / "lnphub_master_v2.0.0.csv",
    Path("/content/lnphub_master_v2.0.0.csv"),
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find lnphub_master_v2.0.0.csv in repo or /content.")

print("Using data file:", DATA_PATH)
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Full LNPHub shape:", df.shape)
display(df.head(3))


In [ ]:
# Cell 3 - Check relevant study/metric counts
required = ["study_id", "experimental_setting", "delivery_metric", "delivery", "lnp_id"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work = df.copy()
work["experimental_setting"] = work["experimental_setting"].astype(str).str.strip()
work["delivery_metric"] = work["delivery_metric"].astype(str).str.strip()
work["study_id"] = work["study_id"].astype(str).str.strip()
work["delivery"] = pd.to_numeric(work["delivery"], errors="coerce")

print("Rows by experimental setting:")
display(work["experimental_setting"].value_counts(dropna=False).reset_index(name="rows").rename(columns={"index": "experimental_setting"}))

print("")
print("Rows by expression-like delivery metric:")
display(
    work[work["delivery_metric"].str.contains("expression", case=False, na=False)]["delivery_metric"]
    .value_counts(dropna=False)
    .reset_index(name="rows")
    .rename(columns={"index": "delivery_metric"})
)


## 2. Select Eligible In Vitro Expression Studies


In [ ]:
# Cell 4 - Filter in vitro expression/expression-discretized studies with >500 formulations
MIN_FORMULATIONS = 500
EXPRESSION_METRICS = {"expression", "expression_discretized", "expression_descritized"}

candidate = work[
    (work["experimental_setting"] == "in_vitro")
    & (work["delivery_metric"].isin(EXPRESSION_METRICS))
    & (work["delivery"].notna())
].copy()

study_summary = (
    candidate.groupby("study_id", dropna=False)
    .agg(
        rows=("study_id", "size"),
        formulations=("lnp_id", "nunique"),
        metrics=("delivery_metric", lambda s: ", ".join(sorted(s.dropna().unique()))),
        delivery_min=("delivery", "min"),
        delivery_median=("delivery", "median"),
        delivery_max=("delivery", "max"),
        delivery_unique_values=("delivery", "nunique"),
    )
    .reset_index()
    .sort_values("formulations", ascending=False)
)

eligible_studies = study_summary.loc[study_summary["formulations"] > MIN_FORMULATIONS, "study_id"].tolist()
subset = candidate[candidate["study_id"].isin(eligible_studies)].copy()

print(f"Candidate in vitro expression rows: {len(candidate):,}")
print(f"Eligible studies with >{MIN_FORMULATIONS:,} unique formulations: {len(eligible_studies):,}")
print(f"Eligible subset rows before 3-class harmonization: {len(subset):,}")

print("")
print("Eligible study summary:")
display(study_summary[study_summary["study_id"].isin(eligible_studies)])


## 3. Harmonize Delivery Into Three Classes


In [ ]:
# Cell 5 - Create harmonized 3-class delivery label: 0=low, 1=medium, 2=high
# Continuous expression uses fixed thresholds on normalized delivery values:
#   low    = delivery < 0
#   medium = 0 <= delivery < 1
#   high   = delivery >= 1
# This does not force equal class sizes; high-delivery LNPs remain rare if they are rare in the data.
CLASS_LABELS = {0: "low", 1: "medium", 2: "high"}
LOW_HIGH_CUTOFF = 0.0
MEDIUM_HIGH_CUTOFF = 1.0


def value_threshold_classes(values):
    values = pd.to_numeric(values, errors="coerce")
    out = pd.Series(pd.NA, index=values.index, dtype="Int64")
    out[values < LOW_HIGH_CUTOFF] = 0
    out[(values >= LOW_HIGH_CUTOFF) & (values < MEDIUM_HIGH_CUTOFF)] = 1
    out[values >= MEDIUM_HIGH_CUTOFF] = 2
    return out


def collapse_discrete_to_three(values):
    values = pd.to_numeric(values, errors="coerce")
    counts = values.value_counts(dropna=True)
    groups = [[float(v)] for v in sorted(counts.index)]

    while len(groups) > 3:
        centers = [np.mean(g) for g in groups]
        distances = [centers[i + 1] - centers[i] for i in range(len(groups) - 1)]
        min_dist = min(distances)
        tied = [i for i, d in enumerate(distances) if np.isclose(d, min_dist)]

        def merged_count(i):
            return sum(int(counts.loc[v]) for v in groups[i] + groups[i + 1])

        merge_i = min(tied, key=merged_count)
        groups[merge_i] = groups[merge_i] + groups[merge_i + 1]
        del groups[merge_i + 1]

    mapping = {}
    for cls, group in enumerate(groups):
        for v in group:
            mapping[v] = cls
    return values.astype(float).map(mapping).astype("Int64"), mapping


class_parts = []
mapping_rows = []

for (study_id, metric), sdf in subset.groupby(["study_id", "delivery_metric"], sort=False):
    sdf = sdf.copy()
    n_unique = sdf["delivery"].nunique(dropna=True)

    if metric in {"expression_discretized", "expression_descritized"}:
        if n_unique == 3:
            ordered = sorted(pd.to_numeric(sdf["delivery"], errors="coerce").dropna().unique())
            mapping = {float(v): i for i, v in enumerate(ordered)}
            sdf["delivery_class_3"] = pd.to_numeric(sdf["delivery"], errors="coerce").astype(float).map(mapping).astype("Int64")
            rule = "already_3_discrete_levels_preserved_order"
        elif n_unique in {4, 5}:
            sdf["delivery_class_3"], mapping = collapse_discrete_to_three(sdf["delivery"])
            rule = f"collapsed_{n_unique}_discrete_levels_to_3_by_closest_adjacent_values"
        else:
            sdf["delivery_class_3"] = value_threshold_classes(sdf["delivery"])
            mapping = {}
            rule = f"unexpected_{n_unique}_discrete_levels_used_fixed_delivery_thresholds"
    else:
        sdf["delivery_class_3"] = value_threshold_classes(sdf["delivery"])
        mapping = {
            "low": f"delivery < {LOW_HIGH_CUTOFF}",
            "medium": f"{LOW_HIGH_CUTOFF} <= delivery < {MEDIUM_HIGH_CUTOFF}",
            "high": f"delivery >= {MEDIUM_HIGH_CUTOFF}",
        }
        rule = "continuous_expression_fixed_delivery_thresholds"

    sdf["delivery_class_label"] = sdf["delivery_class_3"].map(CLASS_LABELS)
    class_parts.append(sdf)
    mapping_rows.append({
        "study_id": study_id,
        "delivery_metric": metric,
        "n_rows": len(sdf),
        "n_formulations": sdf["lnp_id"].nunique(),
        "n_unique_delivery_values": int(n_unique),
        "delivery_mean": sdf["delivery"].mean(),
        "delivery_std": sdf["delivery"].std(),
        "delivery_min": sdf["delivery"].min(),
        "delivery_max": sdf["delivery"].max(),
        "harmonization_rule": rule,
        "discrete_value_to_class_mapping": mapping,
    })

subset_3class = pd.concat(class_parts, ignore_index=True)
mapping_table = pd.DataFrame(mapping_rows)

print("Harmonization rules by study/metric:")
display(mapping_table)

print("")
print("Class sanity check:")
display(
    subset_3class.groupby(["study_id", "delivery_metric", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["study_id", "delivery_metric", "delivery_class_3"])
)


## 4. Save Full 3-Class Subset and Source Tables


In [ ]:
# Cell 6 - Save harmonized subset and summary tables
subset_path = OUT_DIR / "lnphub_in_vitro_expression_gt500formulations_3class.csv"
study_summary_path = OUT_DIR / "study_eligibility_summary.csv"
mapping_path = OUT_DIR / "delivery_3class_harmonization_rules.csv"
class_counts_path = OUT_DIR / "study_class_counts.csv"

class_counts = (
    subset_3class.groupby(["study_id", "delivery_class_3", "delivery_class_label"])
    .agg(rows=("study_id", "size"), formulations=("lnp_id", "nunique"))
    .reset_index()
    .sort_values(["study_id", "delivery_class_3"])
)

subset_3class.to_csv(subset_path, index=False)
study_summary.to_csv(study_summary_path, index=False)
mapping_table.to_csv(mapping_path, index=False)
class_counts.to_csv(class_counts_path, index=False)

print("Saved subset:", subset_path)
print("Saved study summary:", study_summary_path)
print("Saved harmonization rules:", mapping_path)
print("Saved class counts:", class_counts_path)
print("")
print("Final subset shape:", subset_3class.shape)
print("Studies:", subset_3class["study_id"].nunique())
print("Unique formulations:", subset_3class["lnp_id"].nunique())

display(class_counts)


## 5. Balance Classes Within Each Study

This creates a training-ready balanced subset by downsampling majority classes separately within each study. The full 3-class subset remains unchanged.


In [ ]:
# Cell 7 - Balance low / medium / high class counts within each study
# Strategy: for each study, randomly downsample each class to the size of that study's smallest class.
# This keeps class balance study-aware and avoids letting large low-delivery classes dominate training.

BALANCE_RANDOM_SEED = 7
BALANCE_TARGET_MULTIPLIER = 1.0  # 1.0 = exactly the smallest class; change to 1.05 or 1.10 if you want "close" but slightly larger.

rng = np.random.default_rng(BALANCE_RANDOM_SEED)
balanced_parts = []
balance_rows = []

for study_id, sdf in subset_3class.groupby("study_id", sort=False):
    sdf = sdf.dropna(subset=["delivery_class_3"]).copy()
    counts = sdf["delivery_class_3"].value_counts().reindex([0, 1, 2], fill_value=0).astype(int)

    positive_counts = counts[counts > 0]
    if len(positive_counts) < 3:
        balance_rows.append({
            "study_id": study_id,
            "status": "skipped_missing_class",
            "original_low": int(counts.loc[0]),
            "original_medium": int(counts.loc[1]),
            "original_high": int(counts.loc[2]),
            "target_per_class": np.nan,
            "balanced_rows": 0,
            "discarded_rows": int(len(sdf)),
        })
        continue

    min_class_n = int(positive_counts.min())
    target_n = int(np.ceil(min_class_n * BALANCE_TARGET_MULTIPLIER))

    sampled = []
    sampled_counts = {}
    for cls in [0, 1, 2]:
        cls_df = sdf[sdf["delivery_class_3"] == cls]
        take_n = min(len(cls_df), target_n)
        chosen_idx = rng.choice(cls_df.index.to_numpy(), size=take_n, replace=False)
        sampled.append(cls_df.loc[chosen_idx])
        sampled_counts[cls] = take_n

    balanced_study = pd.concat(sampled, ignore_index=False).sample(frac=1, random_state=BALANCE_RANDOM_SEED).reset_index(drop=True)
    balanced_parts.append(balanced_study)

    balance_rows.append({
        "study_id": study_id,
        "status": "balanced",
        "original_low": int(counts.loc[0]),
        "original_medium": int(counts.loc[1]),
        "original_high": int(counts.loc[2]),
        "target_per_class": int(target_n),
        "balanced_low": int(sampled_counts[0]),
        "balanced_medium": int(sampled_counts[1]),
        "balanced_high": int(sampled_counts[2]),
        "original_rows": int(len(sdf)),
        "balanced_rows": int(len(balanced_study)),
        "discarded_rows": int(len(sdf) - len(balanced_study)),
    })

balanced_subset_3class = pd.concat(balanced_parts, ignore_index=True) if balanced_parts else pd.DataFrame(columns=subset_3class.columns)
balanced_subset_3class = balanced_subset_3class.sample(frac=1, random_state=BALANCE_RANDOM_SEED).reset_index(drop=True)
balance_summary = pd.DataFrame(balance_rows)

balanced_class_counts = (
    balanced_subset_3class.groupby(["study_id", "delivery_class_3", "delivery_class_label"])
    .agg(rows=("study_id", "size"), formulations=("lnp_id", "nunique"))
    .reset_index()
    .sort_values(["study_id", "delivery_class_3"])
)

balanced_subset_path = OUT_DIR / "lnphub_in_vitro_expression_gt500formulations_3class_balanced_by_study.csv"
balance_summary_path = OUT_DIR / "study_class_balancing_summary.csv"
balanced_class_counts_path = OUT_DIR / "study_class_counts_balanced.csv"

balanced_subset_3class.to_csv(balanced_subset_path, index=False)
balance_summary.to_csv(balance_summary_path, index=False)
balanced_class_counts.to_csv(balanced_class_counts_path, index=False)

print("Saved balanced subset:", balanced_subset_path)
print("Saved balancing summary:", balance_summary_path)
print("Saved balanced class counts:", balanced_class_counts_path)
print("")
print("Full 3-class subset rows:", f"{len(subset_3class):,}")
print("Balanced subset rows:", f"{len(balanced_subset_3class):,}")
print("Rows discarded during balancing:", f"{len(subset_3class) - len(balanced_subset_3class):,}")
print("Balanced studies:", balanced_subset_3class["study_id"].nunique())

print("")
print("Balancing summary:")
display(balance_summary.sort_values("balanced_rows", ascending=False))

print("")
print("Balanced class counts:")
display(balanced_class_counts)


In [ ]:
# Cell 8 - Plot balanced low / medium / high class counts per study
plot_df = balanced_class_counts.copy()
plot_df["class"] = pd.Categorical(plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True)

balanced_order = (
    balanced_subset_3class.groupby("study_id")["lnp_id"]
    .nunique()
    .sort_values(ascending=False)
    .index
    .tolist()
)

palette = {"low": "#4E79A7", "medium": "#A0A7B4", "high": "#D55E00"}

fig, ax = plt.subplots(figsize=(9.2, max(4.2, 0.34 * len(balanced_order))))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

sns.barplot(
    data=plot_df,
    y="study_id",
    x="rows",
    hue="class",
    order=balanced_order,
    hue_order=["low", "medium", "high"],
    palette=palette,
    ax=ax,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_xlabel("Rows after within-study downsampling")
ax.set_ylabel("")
ax.grid(axis="x", color="#E5E7EB", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(title="3-class delivery", frameon=False, loc="lower right")
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig_path = FIG_DIR / "in_vitro_expression_3class_counts_by_study_balanced.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()


## 6. Hold-Out Split and Molecular Fingerprint / Descriptor Classifiers

This section uses the balanced 3-class subset. `LNPhub_YZ_2024` and `LNPhub_BL_2024` are kept as a completely held-out study-level test set. Every remaining study is then used once as the validation fold in leave-one-study-out CV.


In [ ]:
# Cell 9 - Create held-out test set and leave-one-study-out CV split for 3-class classification
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

CLASS_RANDOM_SEED = 7
HOLDOUT_STUDIES = ["LNPhub_YZ_2024", "LNPhub_BL_2024"]

CLASS_SPLIT_DIR = OUT_DIR / "classification_splits"
CLASS_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

classifier_data = balanced_subset_3class.copy()
classifier_data["study_id"] = classifier_data["study_id"].astype(str).str.strip()
classifier_data["delivery_class_3"] = pd.to_numeric(classifier_data["delivery_class_3"], errors="coerce")
classifier_data = classifier_data[classifier_data["delivery_class_3"].isin([0, 1, 2])].copy()
classifier_data["delivery_class_3"] = classifier_data["delivery_class_3"].astype(int)

missing_holdouts = sorted(set(HOLDOUT_STUDIES) - set(classifier_data["study_id"].unique()))
if missing_holdouts:
    raise ValueError(f"Hold-out studies not found in balanced data: {missing_holdouts}")

test_mask = classifier_data["study_id"].isin(HOLDOUT_STUDIES)
test_classification = classifier_data[test_mask].copy().reset_index(drop=True)
trainval_classification = classifier_data[~test_mask].copy().reset_index(drop=True)

# Leave-one-study-out CV: each remaining study is one validation fold.
trainval_studies = sorted(trainval_classification["study_id"].dropna().unique())
study_to_fold = {study_id: fold for fold, study_id in enumerate(trainval_studies)}
trainval_classification["cv_fold"] = trainval_classification["study_id"].map(study_to_fold).astype(int)

test_classification["cv_fold"] = -1
trainval_classification["oracle_split"] = "trainval"
test_classification["oracle_split"] = "untouched_test"

TRAINVAL_CLASS_PATH = CLASS_SPLIT_DIR / "lnphub_in_vitro_expression_3class_balanced_trainval_leave_one_study_out.csv"
TEST_CLASS_PATH = CLASS_SPLIT_DIR / "lnphub_in_vitro_expression_3class_balanced_untouched_test.csv"
SPLIT_SUMMARY_PATH = CLASS_SPLIT_DIR / "classification_split_summary_leave_one_study_out.csv"

trainval_classification.to_csv(TRAINVAL_CLASS_PATH, index=False)
test_classification.to_csv(TEST_CLASS_PATH, index=False)

split_summary = pd.concat([trainval_classification, test_classification], ignore_index=True)
split_summary_table = (
    split_summary.groupby(["oracle_split", "study_id", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["oracle_split", "study_id", "delivery_class_3"])
)
split_summary_table.to_csv(SPLIT_SUMMARY_PATH, index=False)

print("Saved trainval split:", TRAINVAL_CLASS_PATH)
print("Saved untouched test split:", TEST_CLASS_PATH)
print("Saved split summary:", SPLIT_SUMMARY_PATH)
print("")
print("Trainval rows:", f"{len(trainval_classification):,}")
print("Untouched test rows:", f"{len(test_classification):,}")
print("Trainval studies:", trainval_classification["study_id"].nunique())
print("Untouched test studies:", test_classification["study_id"].nunique(), HOLDOUT_STUDIES)
print("CV strategy: leave-one-study-out")
print("CV folds:", trainval_classification["cv_fold"].nunique())

print("")
print("Rows by split/class:")
display(
    split_summary.groupby(["oracle_split", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
)

print("")
print("Leave-one-study-out fold summary:")
display(
    trainval_classification.groupby("cv_fold")
    .agg(
        validation_study=("study_id", lambda s: s.iloc[0]),
        rows=("study_id", "size"),
        studies=("study_id", "nunique"),
    )
    .reset_index()
)

print("")
print("CV fold x class counts:")
display(
    trainval_classification.groupby(["cv_fold", "study_id", "delivery_class_3"])
    .size()
    .reset_index(name="rows")
    .pivot_table(index=["cv_fold", "study_id"], columns="delivery_class_3", values="rows", fill_value=0)
    .reset_index()
)

print("")
print("Held-out test studies/classes:")
display(
    test_classification.groupby(["study_id", "delivery_class_3", "delivery_class_label"])
    .size()
    .reset_index(name="rows")
    .sort_values(["study_id", "delivery_class_3"])
)


### Molecular Fingerprint / Descriptor Baselines

The first classifier family trains six baselines: ECFP4/RDKit descriptors with MLP, XGBoost, and Random Forest. Metrics are printed live for each leave-one-study-out CV fold and for the held-out test studies.


In [ ]:
# Cell 10 - Train molecular fingerprint / descriptor 3-class classification baselines
# Models: ECFP4 + MLP/XGBoost/Random Forest; RDKit descriptors + MLP/XGBoost/Random Forest
import os
import sys
import json
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from rdkit import Chem
except Exception:
    print("Installing RDKit...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)

try:
    import xgboost
except Exception:
    print("Installing XGBoost...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lnphub_oracles.features import make_ecfp4, make_rdkit_descriptors, smiles_series, valid_smiles_mask

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from xgboost import XGBClassifier

CLASS_LABELS_ORDER = [0, 1, 2]
CLASS_NAMES = {0: "low", 1: "medium", 2: "high"}
CLASS_OUTPUT_DIR = PROJECT_DIR / "results" / "in_vitro_3class_classification_balanced" / "molecular_fingerprint_descriptor"
CLASS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = {
    "ecfp4_mlp": ("ECFP4 + MLP", "ecfp4", "mlp"),
    "ecfp4_xgboost": ("ECFP4 + XGBoost", "ecfp4", "xgboost"),
    "ecfp4_random_forest": ("ECFP4 + Random Forest", "ecfp4", "random_forest"),
    "rdkit_descriptors_mlp": ("RDKit Descriptors + MLP", "rdkit", "mlp"),
    "rdkit_descriptors_xgboost": ("RDKit Descriptors + XGBoost", "rdkit", "xgboost"),
    "rdkit_descriptors_random_forest": ("RDKit Descriptors + Random Forest", "rdkit", "random_forest"),
}


def safe_name(name):
    return name.replace(" + ", "_plus_").replace(" ", "_").replace("/", "_")


def make_classifier(kind, seed):
    if kind == "mlp":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(512, 128),
                alpha=1e-4,
                batch_size=128,
                early_stopping=True,
                max_iter=500,
                random_state=seed,
                verbose=True,
            )),
        ])
    if kind == "random_forest":
        return RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=seed,
            verbose=1,
        )
    if kind == "xgboost":
        return XGBClassifier(
            n_estimators=700,
            max_depth=4,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=3,
            tree_method="hist",
            random_state=seed,
            n_jobs=-1,
            verbosity=1,
        )
    raise ValueError(f"Unknown classifier kind: {kind}")


def fit_classifier(model, kind, X_train, y_train, X_val=None, y_val=None):
    if kind == "xgboost" and X_val is not None and y_val is not None:
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    else:
        model.fit(X_train, y_train)
    return model


def predict_proba_fixed(model, X):
    proba = model.predict_proba(X)
    proba = np.asarray(proba, dtype=float)
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "named_steps"):
        classes = getattr(model.named_steps.get("model"), "classes_", None)
    if classes is None:
        return proba
    fixed = np.zeros((len(X), len(CLASS_LABELS_ORDER)), dtype=float)
    for j, cls in enumerate(classes):
        if int(cls) in CLASS_LABELS_ORDER:
            fixed[:, CLASS_LABELS_ORDER.index(int(cls))] = proba[:, j]
    return fixed


def classification_metrics(y_true, y_pred, proba):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    out = {
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    try:
        out["roc_auc_ovr_macro"] = roc_auc_score(y_true, proba, labels=CLASS_LABELS_ORDER, multi_class="ovr", average="macro")
    except Exception:
        out["roc_auc_ovr_macro"] = np.nan
    try:
        y_bin = label_binarize(y_true, classes=CLASS_LABELS_ORDER)
        out["average_precision_macro"] = average_precision_score(y_bin, proba, average="macro")
    except Exception:
        out["average_precision_macro"] = np.nan
    try:
        out["log_loss"] = log_loss(y_true, proba, labels=CLASS_LABELS_ORDER)
    except Exception:
        out["log_loss"] = np.nan
    return out


def format_metrics(m):
    return (
        f"n={m['n']:,} | acc={m['accuracy']:.3f} | bal_acc={m['balanced_accuracy']:.3f} | "
        f"macroF1={m['macro_f1']:.3f} | AUROC={m['roc_auc_ovr_macro']:.3f} | "
        f"AUPRC={m['average_precision_macro']:.3f} | MCC={m['mcc']:.3f}"
    )


def id_columns(frame):
    return [c for c in ["record_id", "study_id", "lnp_id", "il_id", "il_smiles", "canonical_il_smiles", "delivery", "delivery_class_label"] if c in frame.columns]


trainval = pd.read_csv(TRAINVAL_CLASS_PATH, low_memory=False)
test = pd.read_csv(TEST_CLASS_PATH, low_memory=False)

for frame_name, frame in [("trainval", trainval), ("untouched_test", test)]:
    frame["delivery_class_3"] = pd.to_numeric(frame["delivery_class_3"], errors="coerce")
    before = len(frame)
    frame.dropna(subset=["delivery_class_3"], inplace=True)
    frame["delivery_class_3"] = frame["delivery_class_3"].astype(int)
    print(f"{frame_name}: {before:,} -> {len(frame):,} rows with class labels", flush=True)

train_smiles = smiles_series(trainval)
test_smiles = smiles_series(test)
train_valid = valid_smiles_mask(train_smiles)
test_valid = valid_smiles_mask(test_smiles)

if (~train_valid).sum() or (~test_valid).sum():
    print(f"Dropping invalid SMILES: trainval={(~train_valid).sum():,}, test={(~test_valid).sum():,}", flush=True)

trainval = trainval.loc[train_valid].reset_index(drop=True)
test = test.loc[test_valid].reset_index(drop=True)
train_smiles = smiles_series(trainval)
test_smiles = smiles_series(test)
y = trainval["delivery_class_3"].to_numpy(dtype=int)
y_test = test["delivery_class_3"].to_numpy(dtype=int)

print("Training rows:", f"{len(trainval):,}", flush=True)
print("Held-out test rows:", f"{len(test):,}", flush=True)
print("Training class counts:", dict(pd.Series(y).value_counts().sort_index()), flush=True)
print("Test class counts:", dict(pd.Series(y_test).value_counts().sort_index()), flush=True)

feature_cache = {}
test_feature_cache = {}
manifest_rows = []
all_metric_rows = []

for method_key, (method_name, feature_type, classifier_kind) in METHODS.items():
    print("", flush=True)
    print("=" * 80, flush=True)
    print(method_name, flush=True)
    print("=" * 80, flush=True)

    method_dir = CLASS_OUTPUT_DIR / safe_name(method_name)
    method_dir.mkdir(parents=True, exist_ok=True)

    if feature_type not in feature_cache:
        print(f"Building {feature_type} trainval features...", flush=True)
        feature_cache[feature_type] = make_ecfp4(train_smiles) if feature_type == "ecfp4" else make_rdkit_descriptors(train_smiles)
        print(f"Building {feature_type} untouched-test features...", flush=True)
        test_feature_cache[feature_type] = make_ecfp4(test_smiles) if feature_type == "ecfp4" else make_rdkit_descriptors(test_smiles)

    X, feature_columns = feature_cache[feature_type]
    X_test, test_feature_columns = test_feature_cache[feature_type]
    if feature_columns != test_feature_columns:
        raise ValueError(f"Feature mismatch for {method_name}")

    ok = np.isfinite(X).all(axis=1)
    test_ok = np.isfinite(X_test).all(axis=1)
    X_use = X[ok]
    y_use = y[ok]
    frame_use = trainval.loc[ok].reset_index(drop=True)
    X_test_use = X_test[test_ok]
    y_test_use = y_test[test_ok]
    test_use = test.loc[test_ok].reset_index(drop=True)

    print(f"Feature matrix: trainval={X_use.shape[0]:,} x {X_use.shape[1]:,}; test={X_test_use.shape[0]:,} x {X_test_use.shape[1]:,}", flush=True)

    cv_predictions = []
    metric_rows = []
    folds = sorted(int(f) for f in frame_use["cv_fold"].dropna().unique() if int(f) >= 0)

    for fold in folds:
        val_idx = np.where(frame_use["cv_fold"].astype(int).to_numpy() == fold)[0]
        train_idx = np.setdiff1d(np.arange(len(frame_use)), val_idx)
        X_tr, X_val = X_use[train_idx], X_use[val_idx]
        y_tr, y_val = y_use[train_idx], y_use[val_idx]

        print("", flush=True)
        print(f"{method_name} | fold {fold}: train={len(train_idx):,}, val={len(val_idx):,}", flush=True)
        print("  train classes:", dict(pd.Series(y_tr).value_counts().sort_index()), flush=True)
        print("  val classes:", dict(pd.Series(y_val).value_counts().sort_index()), flush=True)
        print(f"{method_name} | fold {fold}: fitting...", flush=True)

        model = make_classifier(classifier_kind, CLASS_RANDOM_SEED + fold)
        model = fit_classifier(model, classifier_kind, X_tr, y_tr, X_val, y_val)

        print(f"{method_name} | fold {fold}: predicting...", flush=True)
        proba = predict_proba_fixed(model, X_val)
        pred = np.argmax(proba, axis=1)
        metrics = classification_metrics(y_val, pred, proba)
        metrics.update({"method": method_name, "split": f"fold_{fold}", "fold": fold})
        metric_rows.append(metrics)
        all_metric_rows.append(metrics)
        print(f"{method_name} | fold_{fold}: {format_metrics(metrics)}", flush=True)

        pred_df = frame_use.iloc[val_idx][id_columns(frame_use)].copy()
        pred_df["split"] = f"fold_{fold}"
        pred_df["cv_fold"] = fold
        pred_df["y_true"] = y_val
        pred_df["y_pred"] = pred
        for cls in CLASS_LABELS_ORDER:
            pred_df[f"prob_{CLASS_NAMES[cls]}"] = proba[:, cls]
        cv_predictions.append(pred_df)

    cv_df = pd.DataFrame(metric_rows)
    mean_row = cv_df.drop(columns=["method", "split", "fold"], errors="ignore").mean(numeric_only=True).to_dict()
    mean_row.update({"method": method_name, "split": "cv_mean", "fold": -1})
    std_row = cv_df.drop(columns=["method", "split", "fold"], errors="ignore").std(numeric_only=True).to_dict()
    std_row.update({"method": method_name, "split": "cv_std", "fold": -1})
    metric_rows.extend([mean_row, std_row])
    all_metric_rows.extend([mean_row, std_row])
    print("", flush=True)
    print(f"{method_name} | cv_mean: {format_metrics(mean_row)}", flush=True)

    print(f"{method_name}: fitting final model on all trainval rows...", flush=True)
    final_model = make_classifier(classifier_kind, CLASS_RANDOM_SEED)
    final_model = fit_classifier(final_model, classifier_kind, X_use, y_use)
    print(f"{method_name}: final model fit complete", flush=True)

    test_proba = predict_proba_fixed(final_model, X_test_use)
    test_pred = np.argmax(test_proba, axis=1)
    test_metrics = classification_metrics(y_test_use, test_pred, test_proba)
    test_metrics.update({"method": method_name, "split": "untouched_test", "fold": -1})
    metric_rows.append(test_metrics)
    all_metric_rows.append(test_metrics)
    print(f"{method_name} | untouched_test: {format_metrics(test_metrics)}", flush=True)

    test_pred_df = test_use[id_columns(test_use)].copy()
    test_pred_df["split"] = "untouched_test"
    test_pred_df["y_true"] = y_test_use
    test_pred_df["y_pred"] = test_pred
    for cls in CLASS_LABELS_ORDER:
        test_pred_df[f"prob_{CLASS_NAMES[cls]}"] = test_proba[:, cls]

    cm = confusion_matrix(y_test_use, test_pred, labels=CLASS_LABELS_ORDER)
    cm_df = pd.DataFrame(cm, index=[f"true_{CLASS_NAMES[c]}" for c in CLASS_LABELS_ORDER], columns=[f"pred_{CLASS_NAMES[c]}" for c in CLASS_LABELS_ORDER])
    per_class = precision_recall_fscore_support(y_test_use, test_pred, labels=CLASS_LABELS_ORDER, zero_division=0)
    per_class_df = pd.DataFrame({
        "class": [CLASS_NAMES[c] for c in CLASS_LABELS_ORDER],
        "precision": per_class[0],
        "recall": per_class[1],
        "f1": per_class[2],
        "support": per_class[3],
    })

    joblib.dump(final_model, method_dir / "model.joblib")
    pd.concat(cv_predictions, ignore_index=True).to_csv(method_dir / "cv_predictions.csv", index=False)
    test_pred_df.to_csv(method_dir / "untouched_test_predictions.csv", index=False)
    pd.DataFrame(metric_rows).to_csv(method_dir / "metrics.csv", index=False)
    cm_df.to_csv(method_dir / "untouched_test_confusion_matrix.csv")
    per_class_df.to_csv(method_dir / "untouched_test_per_class_metrics.csv", index=False)
    with open(method_dir / "feature_schema.json", "w") as f:
        json.dump({"feature_type": feature_type, "feature_columns": feature_columns, "n_features": len(feature_columns)}, f, indent=2)
    with open(method_dir / "metadata.json", "w") as f:
        json.dump({
            "method": method_name,
            "classifier_kind": classifier_kind,
            "target": "delivery_class_3",
            "class_names": CLASS_NAMES,
            "trainval_path": str(TRAINVAL_CLASS_PATH),
            "untouched_test_path": str(TEST_CLASS_PATH),
            "holdout_studies": HOLDOUT_STUDIES,
            "cv": "leave-one-study-out CV on balanced trainval data",
            "untouched_test_used_for_training": False,
            "output_dir": str(method_dir),
        }, f, indent=2)

    manifest_rows.append({
        "method": method_name,
        "feature_type": feature_type,
        "classifier": classifier_kind,
        "output_dir": str(method_dir),
        "cv_accuracy": mean_row.get("accuracy"),
        "cv_balanced_accuracy": mean_row.get("balanced_accuracy"),
        "cv_macro_f1": mean_row.get("macro_f1"),
        "cv_roc_auc_ovr_macro": mean_row.get("roc_auc_ovr_macro"),
        "cv_average_precision_macro": mean_row.get("average_precision_macro"),
        "test_accuracy": test_metrics.get("accuracy"),
        "test_balanced_accuracy": test_metrics.get("balanced_accuracy"),
        "test_macro_f1": test_metrics.get("macro_f1"),
        "test_roc_auc_ovr_macro": test_metrics.get("roc_auc_ovr_macro"),
        "test_average_precision_macro": test_metrics.get("average_precision_macro"),
    })

manifest = pd.DataFrame(manifest_rows)
all_metrics = pd.DataFrame(all_metric_rows)
manifest_path = CLASS_OUTPUT_DIR / "molecular_fingerprint_descriptor_classification_manifest.csv"
all_metrics_path = CLASS_OUTPUT_DIR / "molecular_fingerprint_descriptor_all_metrics.csv"
manifest.to_csv(manifest_path, index=False)
all_metrics.to_csv(all_metrics_path, index=False)

print("", flush=True)
print("Saved manifest:", manifest_path, flush=True)
print("Saved all metrics:", all_metrics_path, flush=True)
print("Saved models/results under:", CLASS_OUTPUT_DIR, flush=True)
print("")
print("Summary manifest:")
display(manifest)


### Graph Neural Network Baselines

This cell trains the second model family: GCN, GIN, and D-MPNN-style edge-aware GNN classifiers. It uses the same balanced 3-class data, same untouched test studies, and same leave-one-study-out CV split created above.


In [ ]:
# Cell 11 - Train graph neural network 3-class classification baselines
# Models: GCN, GIN, and D-MPNN-style edge-aware GNN
import os
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import torch
    from torch import nn
    import torch.nn.functional as F
    from torch.utils.data import Subset
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GCNConv, GINConv, GINEConv, global_mean_pool
except Exception:
    print("Installing PyTorch Geometric...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"], check=True)
    import torch
    from torch import nn
    import torch.nn.functional as F
    from torch.utils.data import Subset
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GCNConv, GINConv, GINEConv, global_mean_pool

try:
    from rdkit import Chem
except Exception:
    print("Installing RDKit...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)
    from rdkit import Chem

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lnphub_oracles.features import smiles_series

GNN_RANDOM_SEED = 7
GNN_EPOCHS = 120
GNN_PATIENCE = 25
GNN_BATCH_SIZE = 64
GNN_LR = 1e-3
GNN_WEIGHT_DECAY = 1e-5
GNN_EMB_DIM = 128
GNN_LAYERS = 4
GNN_DROPOUT = 0.15

CLASS_LABELS_ORDER = [0, 1, 2]
CLASS_NAMES = {0: "low", 1: "medium", 2: "high"}
GNN_OUTPUT_DIR = PROJECT_DIR / "results" / "in_vitro_3class_classification_balanced" / "graph_neural_network"
GNN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(GNN_RANDOM_SEED)
np.random.seed(GNN_RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, flush=True)

ATOM_LIST = list(range(1, 119))
BOND_LIST = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]


def safe_name(name):
    return name.replace(" + ", "_plus_").replace(" ", "_").replace("/", "_")


def smiles_to_class_graph(smiles, y):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    mol = Chem.AddHs(mol)

    atoms = []
    for atom in mol.GetAtoms():
        atomic_num = atom.GetAtomicNum()
        atoms.append([
            ATOM_LIST.index(atomic_num) if atomic_num in ATOM_LIST else 0,
            min(atom.GetTotalDegree(), 5),
            min(abs(atom.GetFormalCharge()) + 5, 10),
            int(atom.GetIsAromatic()),
        ])

    row, col, edge_feat = [], [], []
    for bond in mol.GetBonds():
        bond_type = bond.GetBondType()
        btype = BOND_LIST.index(bond_type) if bond_type in BOND_LIST else 0
        conj = int(bond.GetIsConjugated())
        ring = int(bond.IsInRing())
        a, b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        row += [a, b]
        col += [b, a]
        edge_feat += [[btype, conj, ring], [btype, conj, ring]]

    if not row:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 3), dtype=torch.long)
    else:
        edge_index = torch.tensor([row, col], dtype=torch.long)
        edge_attr = torch.tensor(edge_feat, dtype=torch.long)

    return Data(
        x=torch.tensor(atoms, dtype=torch.long),
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([int(y)], dtype=torch.long),
    )


def build_class_graphs(frame):
    smiles = smiles_series(frame)
    y = pd.to_numeric(frame["delivery_class_3"], errors="coerce")
    graphs, keep = [], []
    for i, (smi, yi) in enumerate(zip(smiles, y)):
        if not np.isfinite(yi):
            continue
        graph = smiles_to_class_graph(smi, int(yi))
        if graph is None:
            continue
        graph.row_index = int(i)
        graphs.append(graph)
        keep.append(i)
    return graphs, np.asarray(keep, dtype=int)


class AtomEncoder(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.atomic = nn.Embedding(119, emb_dim)
        self.degree = nn.Embedding(6, emb_dim)
        self.charge = nn.Embedding(11, emb_dim)
        self.aromatic = nn.Embedding(2, emb_dim)

    def forward(self, x):
        return self.atomic(x[:, 0]) + self.degree(x[:, 1]) + self.charge(x[:, 2]) + self.aromatic(x[:, 3])


class BondEncoder(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.bond_type = nn.Embedding(4, emb_dim)
        self.conjugated = nn.Embedding(2, emb_dim)
        self.ring = nn.Embedding(2, emb_dim)

    def forward(self, edge_attr):
        return self.bond_type(edge_attr[:, 0]) + self.conjugated(edge_attr[:, 1]) + self.ring(edge_attr[:, 2])


class GCNClassifier(nn.Module):
    def __init__(self, emb_dim=128, layers=4, dropout=0.15, n_classes=3):
        super().__init__()
        self.atom = AtomEncoder(emb_dim)
        self.convs = nn.ModuleList([GCNConv(emb_dim, emb_dim) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(emb_dim, n_classes))

    def forward(self, data):
        h = self.atom(data.x)
        for conv, norm in zip(self.convs, self.norms):
            h = self.dropout(torch.relu(norm(conv(h, data.edge_index))))
        return self.head(global_mean_pool(h, data.batch))


class GINClassifier(nn.Module):
    def __init__(self, emb_dim=128, layers=4, dropout=0.15, n_classes=3):
        super().__init__()
        self.atom = AtomEncoder(emb_dim)
        self.convs = nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU(), nn.Linear(emb_dim, emb_dim))
            self.convs.append(GINConv(mlp))
        self.norms = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(emb_dim, n_classes))

    def forward(self, data):
        h = self.atom(data.x)
        for conv, norm in zip(self.convs, self.norms):
            h = self.dropout(torch.relu(norm(conv(h, data.edge_index))))
        return self.head(global_mean_pool(h, data.batch))


class DMPNNClassifier(nn.Module):
    def __init__(self, emb_dim=128, layers=4, dropout=0.15, n_classes=3):
        super().__init__()
        self.atom = AtomEncoder(emb_dim)
        self.bond = BondEncoder(emb_dim)
        self.convs = nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU(), nn.Linear(emb_dim, emb_dim))
            self.convs.append(GINEConv(mlp))
        self.norms = nn.ModuleList([nn.BatchNorm1d(emb_dim) for _ in range(layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(emb_dim, n_classes))

    def forward(self, data):
        h = self.atom(data.x)
        if data.edge_attr.numel() == 0:
            edge_emb = torch.empty((0, h.shape[1]), device=h.device)
        else:
            edge_emb = self.bond(data.edge_attr)
        for conv, norm in zip(self.convs, self.norms):
            h = self.dropout(torch.relu(norm(conv(h, data.edge_index, edge_emb))))
        return self.head(global_mean_pool(h, data.batch))


GNN_MODELS = {
    "gcn": ("GCN", GCNClassifier),
    "gin": ("GIN", GINClassifier),
    "dmpnn": ("D-MPNN", DMPNNClassifier),
}


def make_gnn(model_class):
    return model_class(emb_dim=GNN_EMB_DIM, layers=GNN_LAYERS, dropout=GNN_DROPOUT, n_classes=3).to(device)


def class_weights_from_y(y_train):
    counts = np.bincount(np.asarray(y_train, dtype=int), minlength=3).astype(float)
    weights = counts.sum() / (len(counts) * np.maximum(counts, 1.0))
    return torch.tensor(weights, dtype=torch.float32, device=device)


def evaluate_gnn(model, loader):
    model.eval()
    ys, probs = [], []
    with torch.inference_mode():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            prob = F.softmax(logits, dim=1)
            ys.append(batch.y.view(-1).detach().cpu().numpy())
            probs.append(prob.detach().cpu().numpy())
    y_true = np.concatenate(ys).astype(int)
    proba = np.vstack(probs)
    y_pred = np.argmax(proba, axis=1)
    return y_true, y_pred, proba


def classification_metrics(y_true, y_pred, proba):
    out = {
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    try:
        out["roc_auc_ovr_macro"] = roc_auc_score(y_true, proba, labels=CLASS_LABELS_ORDER, multi_class="ovr", average="macro")
    except Exception:
        out["roc_auc_ovr_macro"] = np.nan
    try:
        y_bin = label_binarize(y_true, classes=CLASS_LABELS_ORDER)
        out["average_precision_macro"] = average_precision_score(y_bin, proba, average="macro")
    except Exception:
        out["average_precision_macro"] = np.nan
    try:
        out["log_loss"] = log_loss(y_true, proba, labels=CLASS_LABELS_ORDER)
    except Exception:
        out["log_loss"] = np.nan
    return out


def format_metrics(m):
    return (
        f"n={m['n']:,} | acc={m['accuracy']:.3f} | bal_acc={m['balanced_accuracy']:.3f} | "
        f"macroF1={m['macro_f1']:.3f} | AUROC={m['roc_auc_ovr_macro']:.3f} | "
        f"AUPRC={m['average_precision_macro']:.3f} | MCC={m['mcc']:.3f}"
    )


def id_columns(frame):
    return [c for c in ["record_id", "study_id", "lnp_id", "il_id", "il_smiles", "canonical_il_smiles", "delivery", "delivery_class_label"] if c in frame.columns]


def train_one_fold(model, graphs, train_idx, val_idx, y_train, model_name, fold_label):
    train_loader = DataLoader(Subset(graphs, train_idx.tolist()), batch_size=GNN_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(Subset(graphs, val_idx.tolist()), batch_size=GNN_BATCH_SIZE, shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=GNN_LR, weight_decay=GNN_WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_from_y(y_train))

    best_state = None
    best_val_loss = float("inf")
    bad_epochs = 0

    for epoch in range(1, GNN_EPOCHS + 1):
        model.train()
        train_losses = []
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch)
            loss = loss_fn(logits, batch.y.view(-1))
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.inference_mode():
            for batch in val_loader:
                batch = batch.to(device)
                logits = model(batch)
                loss = loss_fn(logits, batch.y.view(-1))
                val_losses.append(float(loss.detach().cpu()))

        train_loss = float(np.mean(train_losses)) if train_losses else np.nan
        val_loss = float(np.mean(val_losses)) if val_losses else np.inf

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch == 1 or epoch % 10 == 0 or bad_epochs == GNN_PATIENCE:
            print(f"{model_name} | {fold_label} | epoch {epoch:03d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, best_val={best_val_loss:.4f}", flush=True)

        if bad_epochs >= GNN_PATIENCE:
            print(f"{model_name} | {fold_label}: early stopping at epoch {epoch}", flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return train_loader, val_loader, best_val_loss


def save_predictions(frame_part, split, y_true, y_pred, proba, cv_fold=None):
    out = frame_part[id_columns(frame_part)].copy()
    out["split"] = split
    if cv_fold is not None:
        out["cv_fold"] = cv_fold
    out["y_true"] = y_true
    out["y_pred"] = y_pred
    for cls in CLASS_LABELS_ORDER:
        out[f"prob_{CLASS_NAMES[cls]}"] = proba[:, cls]
    return out


trainval_gnn = pd.read_csv(TRAINVAL_CLASS_PATH, low_memory=False)
test_gnn = pd.read_csv(TEST_CLASS_PATH, low_memory=False)
for frame in [trainval_gnn, test_gnn]:
    frame["delivery_class_3"] = pd.to_numeric(frame["delivery_class_3"], errors="coerce")
    frame.dropna(subset=["delivery_class_3"], inplace=True)
    frame["delivery_class_3"] = frame["delivery_class_3"].astype(int)

print("Building trainval molecular graphs...", flush=True)
train_graphs, train_keep = build_class_graphs(trainval_gnn)
trainval_gnn = trainval_gnn.iloc[train_keep].reset_index(drop=True)
y_gnn = trainval_gnn["delivery_class_3"].to_numpy(dtype=int)
print(f"Trainval graphs: {len(train_graphs):,}; failed/skipped: {len(pd.read_csv(TRAINVAL_CLASS_PATH, low_memory=False)) - len(train_graphs):,}", flush=True)

print("Building untouched-test molecular graphs...", flush=True)
test_graphs, test_keep = build_class_graphs(test_gnn)
test_gnn = test_gnn.iloc[test_keep].reset_index(drop=True)
y_test_gnn = test_gnn["delivery_class_3"].to_numpy(dtype=int)
print(f"Untouched-test graphs: {len(test_graphs):,}; failed/skipped: {len(pd.read_csv(TEST_CLASS_PATH, low_memory=False)) - len(test_graphs):,}", flush=True)

print("Trainval class counts:", dict(pd.Series(y_gnn).value_counts().sort_index()), flush=True)
print("Untouched-test class counts:", dict(pd.Series(y_test_gnn).value_counts().sort_index()), flush=True)

manifest_rows = []
all_metric_rows = []

for model_key, (model_name, model_class) in GNN_MODELS.items():
    print("", flush=True)
    print("=" * 80, flush=True)
    print(model_name, flush=True)
    print("=" * 80, flush=True)

    out_dir = GNN_OUTPUT_DIR / safe_name(model_name)
    out_dir.mkdir(parents=True, exist_ok=True)
    cv_predictions = []
    metric_rows = []

    folds = sorted(int(f) for f in trainval_gnn["cv_fold"].dropna().unique() if int(f) >= 0)
    for fold in folds:
        val_idx = np.where(trainval_gnn["cv_fold"].astype(int).to_numpy() == fold)[0]
        train_idx = np.setdiff1d(np.arange(len(trainval_gnn)), val_idx)
        val_study = ", ".join(sorted(trainval_gnn.iloc[val_idx]["study_id"].unique()))
        y_train = y_gnn[train_idx]

        print("", flush=True)
        print(f"{model_name} | fold {fold}: validation study = {val_study}", flush=True)
        print(f"{model_name} | fold {fold}: train={len(train_idx):,}, val={len(val_idx):,}", flush=True)
        print("  train classes:", dict(pd.Series(y_train).value_counts().sort_index()), flush=True)
        print("  val classes:", dict(pd.Series(y_gnn[val_idx]).value_counts().sort_index()), flush=True)

        model = make_gnn(model_class)
        _, val_loader, best_val_loss = train_one_fold(model, train_graphs, train_idx, val_idx, y_train, model_name, f"fold_{fold}")
        y_true, y_pred, proba = evaluate_gnn(model, val_loader)
        metrics = classification_metrics(y_true, y_pred, proba)
        metrics.update({"method": model_name, "split": f"fold_{fold}", "fold": fold, "validation_study": val_study, "best_val_loss": best_val_loss})
        metric_rows.append(metrics)
        all_metric_rows.append(metrics)
        print(f"{model_name} | fold_{fold}: {format_metrics(metrics)}", flush=True)

        cv_predictions.append(save_predictions(trainval_gnn.iloc[val_idx], f"fold_{fold}", y_true, y_pred, proba, cv_fold=fold))

    fold_metrics_df = pd.DataFrame(metric_rows)
    mean_row = fold_metrics_df.drop(columns=["method", "split", "fold", "validation_study"], errors="ignore").mean(numeric_only=True).to_dict()
    mean_row.update({"method": model_name, "split": "cv_mean", "fold": -1, "validation_study": "leave_one_study_out"})
    std_row = fold_metrics_df.drop(columns=["method", "split", "fold", "validation_study"], errors="ignore").std(numeric_only=True).to_dict()
    std_row.update({"method": model_name, "split": "cv_std", "fold": -1, "validation_study": "leave_one_study_out"})
    metric_rows.extend([mean_row, std_row])
    all_metric_rows.extend([mean_row, std_row])
    print("", flush=True)
    print(f"{model_name} | cv_mean: {format_metrics(mean_row)}", flush=True)

    print(f"{model_name}: fitting final model on all trainval graphs...", flush=True)
    final_model = make_gnn(model_class)
    all_idx = np.arange(len(train_graphs))
    train_one_fold(final_model, train_graphs, all_idx, all_idx, y_gnn, model_name, "final_trainval")
    print(f"{model_name}: final model fit complete", flush=True)

    test_loader = DataLoader(test_graphs, batch_size=GNN_BATCH_SIZE, shuffle=False)
    y_true, y_pred, proba = evaluate_gnn(final_model, test_loader)
    test_metrics = classification_metrics(y_true, y_pred, proba)
    test_metrics.update({"method": model_name, "split": "untouched_test", "fold": -1, "validation_study": ", ".join(HOLDOUT_STUDIES)})
    metric_rows.append(test_metrics)
    all_metric_rows.append(test_metrics)
    print(f"{model_name} | untouched_test: {format_metrics(test_metrics)}", flush=True)

    test_predictions = save_predictions(test_gnn, "untouched_test", y_true, y_pred, proba)
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_LABELS_ORDER)
    cm_df = pd.DataFrame(cm, index=[f"true_{CLASS_NAMES[c]}" for c in CLASS_LABELS_ORDER], columns=[f"pred_{CLASS_NAMES[c]}" for c in CLASS_LABELS_ORDER])
    per_class = precision_recall_fscore_support(y_true, y_pred, labels=CLASS_LABELS_ORDER, zero_division=0)
    per_class_df = pd.DataFrame({
        "class": [CLASS_NAMES[c] for c in CLASS_LABELS_ORDER],
        "precision": per_class[0],
        "recall": per_class[1],
        "f1": per_class[2],
        "support": per_class[3],
    })

    torch.save({
        "state_dict": final_model.state_dict(),
        "model_key": model_key,
        "model_name": model_name,
        "model_class": model_class.__name__,
        "model_params": {
            "emb_dim": GNN_EMB_DIM,
            "layers": GNN_LAYERS,
            "dropout": GNN_DROPOUT,
            "n_classes": 3,
        },
        "class_names": CLASS_NAMES,
    }, out_dir / "model.pt")
    pd.concat(cv_predictions, ignore_index=True).to_csv(out_dir / "cv_predictions.csv", index=False)
    test_predictions.to_csv(out_dir / "untouched_test_predictions.csv", index=False)
    pd.DataFrame(metric_rows).to_csv(out_dir / "metrics.csv", index=False)
    cm_df.to_csv(out_dir / "untouched_test_confusion_matrix.csv")
    per_class_df.to_csv(out_dir / "untouched_test_per_class_metrics.csv", index=False)
    with open(out_dir / "metadata.json", "w") as f:
        json.dump({
            "method": model_name,
            "model_key": model_key,
            "target": "delivery_class_3",
            "class_names": CLASS_NAMES,
            "trainval_path": str(TRAINVAL_CLASS_PATH),
            "untouched_test_path": str(TEST_CLASS_PATH),
            "holdout_studies": HOLDOUT_STUDIES,
            "cv": "leave-one-study-out CV on balanced trainval data",
            "untouched_test_used_for_training": False,
            "epochs_max": GNN_EPOCHS,
            "early_stopping_patience": GNN_PATIENCE,
            "batch_size": GNN_BATCH_SIZE,
            "lr": GNN_LR,
            "weight_decay": GNN_WEIGHT_DECAY,
            "graph_featurization": "RDKit atom/bond graph from ionizable-lipid SMILES",
            "output_dir": str(out_dir),
        }, f, indent=2)

    manifest_rows.append({
        "method": model_name,
        "output_dir": str(out_dir),
        "cv_accuracy": mean_row.get("accuracy"),
        "cv_balanced_accuracy": mean_row.get("balanced_accuracy"),
        "cv_macro_f1": mean_row.get("macro_f1"),
        "cv_roc_auc_ovr_macro": mean_row.get("roc_auc_ovr_macro"),
        "cv_average_precision_macro": mean_row.get("average_precision_macro"),
        "test_accuracy": test_metrics.get("accuracy"),
        "test_balanced_accuracy": test_metrics.get("balanced_accuracy"),
        "test_macro_f1": test_metrics.get("macro_f1"),
        "test_roc_auc_ovr_macro": test_metrics.get("roc_auc_ovr_macro"),
        "test_average_precision_macro": test_metrics.get("average_precision_macro"),
    })

manifest = pd.DataFrame(manifest_rows)
all_metrics = pd.DataFrame(all_metric_rows)
manifest_path = GNN_OUTPUT_DIR / "gnn_classification_manifest.csv"
all_metrics_path = GNN_OUTPUT_DIR / "gnn_all_metrics.csv"
manifest.to_csv(manifest_path, index=False)
all_metrics.to_csv(all_metrics_path, index=False)

print("", flush=True)
print("Saved GNN manifest:", manifest_path, flush=True)
print("Saved all GNN metrics:", all_metrics_path, flush=True)
print("Saved GNN models/results under:", GNN_OUTPUT_DIR, flush=True)
print("")
print("GNN summary manifest:")
display(manifest)


## 7. Plot Per-Study Class Counts


In [ ]:
# Cell 13 - Plot full unbalanced low / medium / high class counts per study
plot_df = class_counts.copy()
plot_df["class"] = pd.Categorical(plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True)

order = (
    subset_3class.groupby("study_id")["lnp_id"]
    .nunique()
    .sort_values(ascending=False)
    .index
    .tolist()
)

palette = {"low": "#4E79A7", "medium": "#A0A7B4", "high": "#D55E00"}

fig, ax = plt.subplots(figsize=(9.2, max(4.2, 0.34 * len(order))))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

sns.barplot(
    data=plot_df,
    y="study_id",
    x="rows",
    hue="class",
    order=order,
    hue_order=["low", "medium", "high"],
    palette=palette,
    ax=ax,
    edgecolor="white",
    linewidth=0.5,
)

ax.set_xlabel("Rows")
ax.set_ylabel("")
ax.grid(axis="x", color="#E5E7EB", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(title="3-class delivery", frameon=False, loc="lower right")
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig_path = FIG_DIR / "in_vitro_expression_3class_counts_by_study.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()


In [ ]:
# Cell 14 - Faceted full unbalanced per-study class-count view
plot_df = subset_3class.copy()
plot_df["delivery_class_label"] = pd.Categorical(
    plot_df["delivery_class_label"], categories=["low", "medium", "high"], ordered=True
)

n_studies = plot_df["study_id"].nunique()
ncols = 3
nrows = int(np.ceil(n_studies / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(9.2, max(3.2, 2.05 * nrows)), squeeze=False)
fig.patch.set_facecolor("white")

for ax in axes.flat:
    ax.axis("off")

for ax, study_id in zip(axes.flat, order):
    sdf = plot_df[plot_df["study_id"] == study_id]
    counts = sdf["delivery_class_label"].value_counts().reindex(["low", "medium", "high"], fill_value=0)
    ax.axis("on")
    ax.bar(counts.index.astype(str), counts.values, color=[palette[k] for k in counts.index], width=0.72)
    ax.set_title(study_id, fontsize=9, loc="left", pad=3)
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(axis="y", color="#E5E7EB", linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for x, y in enumerate(counts.values):
        ax.text(x, y, f"{int(y):,}", ha="center", va="bottom", fontsize=7)

fig.tight_layout(h_pad=1.0, w_pad=1.0)
fig_path = FIG_DIR / "in_vitro_expression_3class_counts_faceted.pdf"
fig.savefig(fig_path, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()
